# Tuần 2 — Standard RAG Generator (Hệ 2 trong `PIPELINE.md`)

Notebook này build **Hệ 2 — Standard RAG** (theo kiến trúc ở `final/PIPELINE.md` §3): retrieve top-k cố định rồi generate thẳng, **chưa có** retrieval-decision/ISREL/ISSUP/ISUSE — 4 module đó là Notebook 3 (Self-RAG-inspired).

**Yêu cầu trước khi chạy**: đã chạy xong `01_retrieval_baseline.ipynb` — notebook này load lại `chunks.faiss`, `chunks_meta.json`, `dev_split_qids.json` từ `ARTIFACT_DIR`, không build lại từ đầu.

Output: `standard_rag_results.jsonl` trong `ARTIFACT_DIR` (mỗi dòng 1 câu hỏi: câu trả lời sinh ra + nguồn trích dẫn + câu trả lời gold) — dùng làm input cho bảng so sánh 3 hệ ở Notebook 4.

## 0. Cấu hình môi trường + API key

- Trên Colab: mount Drive **chỉ để lấy dữ liệu/artifact** (không `git clone` vào Drive — xem lý do ở `PIPELINE.md` §6.4). Mở notebook này trực tiếp từ GitHub, không clone.
- **API key Groq** (miễn phí, không cần khai báo billing): lấy tại https://console.groq.com/keys . Trên Colab, bấm biểu tượng chìa khóa 🔑 ở sidebar trái → "Add new secret" → tên `GROQ_API_KEY`, dán key vào, bật "Notebook access". Chạy local: đặt biến môi trường `GROQ_API_KEY`, hoặc notebook sẽ hỏi nhập trực tiếp (không hardcode key vào cell).
- *(Ghi chú: ban đầu dự định dùng Gemini, nhưng tài khoản của bạn bị bắt setup billing để lấy key — Groq free tier hiện không yêu cầu việc này.)*

In [ ]:
!pip install -q sentence-transformers faiss-cpu groq

In [ ]:
import os

try:
    IN_COLAB = "google.colab" in str(get_ipython())
except NameError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DATA_ROOT = "/content/drive/MyDrive/NLP-CS2308.CH203-data"  # sua neu ban dat ten khac
    DATA_DIR = os.path.join(DRIVE_DATA_ROOT, "VLQA")
    ARTIFACT_DIR = os.path.join(DRIVE_DATA_ROOT, "artifacts")

    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
else:
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
    DATA_DIR = os.path.join(REPO_DIR, "final", "dataset", "VLQA")
    ARTIFACT_DIR = os.path.join(REPO_DIR, "final", "artifacts")
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    import getpass
    GROQ_API_KEY = getpass.getpass("Nhap GROQ_API_KEY: ")

print("DATA_DIR:", DATA_DIR, "| exists:", os.path.isdir(DATA_DIR))
print("ARTIFACT_DIR:", ARTIFACT_DIR, "| exists:", os.path.isdir(ARTIFACT_DIR))
print("GROQ_API_KEY loaded:", bool(GROQ_API_KEY))

## 1. Load artifact từ Notebook 1 (index, chunk metadata, dev split)

In [ ]:
import json
import faiss

INDEX_PATH = os.path.join(ARTIFACT_DIR, "chunks.faiss")
CHUNKS_META_PATH = os.path.join(ARTIFACT_DIR, "chunks_meta.json")
SPLIT_PATH = os.path.join(ARTIFACT_DIR, "dev_split_qids.json")

for path in [INDEX_PATH, CHUNKS_META_PATH, SPLIT_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Khong tim thay {path} - hay chay 01_retrieval_baseline.ipynb truoc")

index = faiss.read_index(INDEX_PATH)
with open(CHUNKS_META_PATH, encoding="utf-8") as f:
    chunks = json.load(f)
with open(SPLIT_PATH, encoding="utf-8") as f:
    dev_qids = set(json.load(f))

with open(os.path.join(DATA_DIR, "train.json"), encoding="utf-8") as f:
    train_full = json.load(f)
dev_set = [ex for ex in train_full if ex["qid"] in dev_qids]

print(f"Da load index: {index.ntotal} chunk | dev set: {len(dev_set)} cau")

## 2. Embedding model + hàm `retrieve` (giữ nguyên tinh thần Notebook 1)

Notebook này chạy độc lập (không import chéo giữa các notebook) nên định nghĩa lại `retrieve`, lần này trả về nguyên `chunk` (cần `text` + `law_id` để build context cho prompt) thay vì chỉ `aid`.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "bkai-foundation-models/vietnamese-bi-encoder"
device = "cuda" if torch.cuda.is_available() else "cpu"
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device=device)
print("Embedding device:", device)


def retrieve(query, top_chunks=50, max_k=5):
    q_emb = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    _, idxs = index.search(q_emb, top_chunks)

    results, seen = [], set()
    for idx in idxs[0]:
        c = chunks[idx]
        if c["aid"] not in seen:
            seen.add(c["aid"])
            results.append(c)
        if len(results) >= max_k:
            break
    return results

## 3. Groq client

Model khả dụng khác nhau theo từng tài khoản/thời điểm (đã gặp thật: `llama-3.3-70b-versatile` báo `model_not_found` dù tài liệu Groq liệt kê là production) — nên **không hardcode một tên model duy nhất**, mà hỏi thẳng API xem tài khoản này thực sự dùng được model nào, rồi chọn theo thứ tự ưu tiên.

In [ ]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

available_models = sorted(m.id for m in client.models.list().data)
print("Model kha dung tren tai khoan nay:")
for m in available_models:
    print(" -", m)

CANDIDATE_MODELS = [
    "llama-3.3-70b-versatile",
    "openai/gpt-oss-120b",
    "llama-3.1-8b-instant",
    "openai/gpt-oss-20b",
]
GENERATOR_MODEL = next((m for m in CANDIDATE_MODELS if m in available_models), None) or available_models[0]
print("\nDang dung GENERATOR_MODEL =", GENERATOR_MODEL)

## 4. Prompt template Standard RAG

Nguyên tắc: chỉ trả lời dựa trên context được cung cấp, ép model từ chối/nói rõ "không đủ căn cứ" thay vì suy đoán khi context không đủ, và bắt buộc trích dẫn `law_id` — để Notebook 3 (ISSUP) có thể kiểm tra được câu trả lời có bám evidence hay không.

**Về rate limit**: Groq free tier giới hạn cả request/phút lẫn token/phút, và số cụ thể thay đổi theo model/tài khoản nên không hardcode ở đây — cách bền hơn là đọc thẳng header `Retry-After` mà server trả về khi bị 429 rồi chờ đúng từng đó giây, thay vì đoán mù bằng backoff cố định. Nếu vẫn bị rate-limit liên tục, xem gợi ý ở cell bên dưới (đổi model, giảm `max_k`, tăng `SLEEP_BETWEEN_CALLS`).

In [ ]:
import time
from groq import RateLimitError, APIStatusError

PROMPT_TEMPLATE = """Ban la tro ly tu van phap luat Viet Nam. Chi tra loi dua tren cac dieu luat duoc cung cap ben duoi. Neu cac dieu luat khong du thong tin de tra loi, hay noi ro la khong du can cu thay vi suy doan. Khi tra loi, trich dan van ban luat tuong ung bang ky hieu [so] va ghi ro ma so van ban.

Cac dieu luat lien quan:
{context}

Cau hoi: {question}

Tra loi (tieng Viet, ngan gon, co trich dan):"""


def build_context(retrieved):
    parts = []
    for i, c in enumerate(retrieved, start=1):
        parts.append(f"[{i}] (Van ban: {c['law_id']})\n{c['text']}")
    return "\n\n".join(parts)


def retry_after_seconds(exc, fallback):
    response = getattr(exc, "response", None)
    header = response.headers.get("retry-after") if response is not None else None
    if header is None:
        return fallback
    try:
        return float(header)
    except ValueError:
        return fallback


def generate_answer(question, retrieved, max_retries=6):
    prompt = PROMPT_TEMPLATE.format(context=build_context(retrieved), question=question)
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=GENERATOR_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.2,
            )
            return (response.choices[0].message.content or "").strip()
        except RateLimitError as e:
            wait = retry_after_seconds(e, fallback=2 ** attempt)
            print(f"Rate limit (lan {attempt + 1}/{max_retries}) -> cho {wait:.1f}s theo Retry-After tu server")
            time.sleep(wait)
        except APIStatusError as e:
            if e.status_code >= 500:
                wait = 2 ** attempt
                print(f"Loi server {e.status_code} (lan {attempt + 1}/{max_retries}) -> cho {wait}s")
                time.sleep(wait)
            else:
                print(f"Loi khong the retry duoc (status {e.status_code}): {e}")
                return ""
        except Exception as e:
            wait = 2 ** attempt
            print(f"Loi khac (lan {attempt + 1}/{max_retries}): {e} -> cho {wait}s")
            time.sleep(wait)
    return ""

## 5. Chạy generation trên dev set (checkpoint từng câu, resume-safe)

Ghi thẳng từng kết quả vào file JSONL và `flush()` ngay — nếu Colab bị ngắt session giữa chừng, chạy lại cell này sẽ tự bỏ qua các `qid` đã có, không mất tiến độ. `MAX_QUESTIONS` đặt số nguyên để chạy thử nhanh (ví dụ 20), để `None` để chạy full dev set.

Ba tay đòn giảm rate-limit nếu vẫn bị 429 liên tục dù đã retry theo `Retry-After`:

1. **`SLEEP_BETWEEN_CALLS`** — tăng lên (ví dụ 3-5s) để chủ động giãn request thay vì để đến khi bị chặn mới chờ.
2. **`RETRIEVE_K`** — giảm số passage đưa vào context (ví dụ 3 thay vì 5) → giảm token/request, đỡ áp lực lên giới hạn token/phút.
3. **`GENERATOR_MODEL`** — model nhỏ hơn (`llama-3.1-8b-instant`) thường có hạn mức free-tier cao hơn model 70B; đổi thứ tự ưu tiên trong `CANDIDATE_MODELS` ở cell Groq client nếu muốn ưu tiên model nhỏ.

Nếu bị chặn ở mức **quota theo ngày** (không phải theo phút) thì không cách nào chờ được trong phiên hiện tại — hạ `MAX_QUESTIONS` để chạy từng đợt nhỏ trải ra nhiều ngày, tận dụng cơ chế resume-safe.

In [ ]:
MAX_QUESTIONS = None
SLEEP_BETWEEN_CALLS = 2  # tang len neu van bi 429 lien tuc
RETRIEVE_K = 5  # giam xuong 3 neu muon giam token/request
RESULTS_PATH = os.path.join(ARTIFACT_DIR, "standard_rag_results.jsonl")

existing_records = []
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            if rec["generated_answer"]:
                existing_records.append(rec)
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        for rec in existing_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

done_qids = {rec["qid"] for rec in existing_records}
print(f"Da co san {len(done_qids)} cau tra loi thanh cong tu lan chay truoc (da loai bo cac lan loi/rong)")

questions_to_run = dev_set if MAX_QUESTIONS is None else dev_set[:MAX_QUESTIONS]

with open(RESULTS_PATH, "a", encoding="utf-8") as f:
    for ex in questions_to_run:
        if ex["qid"] in done_qids:
            continue
        retrieved = retrieve(ex["question"], top_chunks=50, max_k=RETRIEVE_K)
        answer = generate_answer(ex["question"], retrieved)
        record = {
            "qid": ex["qid"],
            "question": ex["question"],
            "gold_answer": ex["answer"],
            "retrieved_aids": [c["aid"] for c in retrieved],
            "retrieved_law_ids": [c["law_id"] for c in retrieved],
            "generated_answer": answer,
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        f.flush()
        time.sleep(SLEEP_BETWEEN_CALLS)

print("Hoan tat.")

## 6. Soi vài kết quả (định tính)

So sánh câu trả lời sinh ra với câu trả lời gold — đánh giá định lượng thật (ISSUP/ISUSE) sẽ làm ở Notebook 3+4, ở đây chỉ xem nhanh chất lượng bằng mắt.

In [ ]:
import pandas as pd

results_df = pd.read_json(RESULTS_PATH, lines=True)
print(f"So cau da sinh: {len(results_df)}")

for _, row in results_df.sample(min(3, len(results_df)), random_state=0).iterrows():
    print("qid:", row["qid"])
    print("Cau hoi:", row["question"])
    print("Nguon trich dan (law_id):", row["retrieved_law_ids"])
    print("Answer sinh ra:", row["generated_answer"])
    print("Gold answer:", row["gold_answer"])
    print("-" * 80)

## 7. Bước tiếp theo

- **Notebook 3**: thêm 4 module reflection (Retrieve-decision, ISREL, ISSUP, ISUSE) lên trên cùng `retrieve()` + `generate_answer()` ở đây, tạo `self_rag_results.jsonl` cùng schema để dễ so sánh.
- **Notebook 4**: load cả `standard_rag_results.jsonl` và `self_rag_results.jsonl`, chấm ISSUP/ISUSE bằng LLM-judge, xuất `final_comparison_table.csv`.